# European option collars

This notebook prices a strip of **12 monthly collars**. Each collar is:

- a **purchased put** (floor)
- a **sold call** (cap)

Python owns the maturity and settlement dates. The binding converts `datetime.date` values internally.

`EuropeanOption.get_option_value` is the unsigned Black-76 premium times volume. `get_value` applies the buy/sell sign and subtracts the PV of the contracted premium. The collars here have a contracted premium of zero, so `get_value` is just the signed option value.

Build the module before running the notebook (stages `sjd.pyd` into `python/sjd_runtime`):

`cmake --build --preset debug --target sjd_python`

Shut down the notebook kernel before rebuilding so Windows can overwrite `sjd.pyd`.

In [1]:
import math
import sys
from datetime import date
from pathlib import Path

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _notebooks = _candidate / "python" / "notebooks"
    if (_notebooks / "sjd_setup.py").is_file():
        if str(_notebooks) not in sys.path:
            sys.path.insert(0, str(_notebooks))
        break
    if (_candidate / "sjd_setup.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise RuntimeError("Could not find python/notebooks/sjd_setup.py from " + str(_cwd))

import sjd_setup

sjd = sjd_setup.load_sjd()
print(sjd.__file__)
print([name for name in dir(sjd) if not name.startswith("_")])


E:\Code\sjd\python\sjd_runtime\sjd.pyd
['AsianForward', 'AsianOption', 'Black76Call', 'Black76Put', 'BusinessDayConvention', 'BuySell', 'Calendar', 'CurveInterpolator', 'DailyFixingCurve', 'DiscountCurve', 'EuropeanForward', 'EuropeanOption', 'ForwardAveraging', 'ForwardCurve', 'ForwardRoll', 'Margin', 'Portfolio', 'PortfolioB2CEM', 'PortfolioEvolver', 'PutCall', 'RatesEnvironment', 'RegCapCreditParameters', 'RegCapUnderlyingFactors', 'RegulatoryCapital', 'SimpleMargin', 'Simulation', 'SingleFactorEconomicScenarioGenerator', 'SurfaceInterpolator', 'TimeUnit', 'ValidationIssue', 'ValidationReport', 'ValidationSeverity', 'VolatilityCurve', 'VolatilityCurveCMA', 'VolatilityRateSource', 'VolatilitySurfaceDelta', 'VolatilitySurfaceMoneynessCMA', 'reg_cap_factors', 'simulate']


## Market data

The example quote is domestic currency units per unit of foreign currency. Forward rates are generated from flat domestic and foreign continuously compounded rates; discount factors use the domestic rate. Volatility is flat.

Curve pillars are the valuation date plus each collar settlement, so no interpolation or extrapolation is required.

Black-76 uses:

- forward and discount factor on the **settlement** date
- standard deviation to the **maturity** date (`vol * sqrt(Actual/365)`)

In [2]:
anchor = sjd_setup.ANCHOR
holidays = sjd_setup.sample_calendar(sjd)
settlements = [holidays.adjust(sjd_setup.add_months(anchor, i)) for i in range(1, 13)]
curve_dates = [anchor, *settlements]
yf = sjd_setup.year_fraction

volume = 1_000_000.0
put_moneyness = 0.97
call_moneyness = 1.03

frs, drs, vrs = sjd_setup.fx_curves(sjd, curve_dates)

print("curves ok:", frs.is_ok(), drs.is_ok(), vrs.is_ok())
print("settlements:", ", ".join(d.isoformat() for d in settlements))


curves ok: True True True
settlements: 2026-10-16, 2026-11-16, 2026-12-17, 2027-01-19, 2027-02-16, 2027-03-16, 2027-04-16, 2027-05-17, 2027-06-17, 2027-07-16, 2027-08-16, 2027-09-16


In [3]:
environment = sjd.RatesEnvironment(anchor)
environment.set_frs(frs)
environment.set_drs(drs)
environment.set_vrs(vrs)

print("environment:", environment.is_ok(), environment.error_messages())

environment: True 


## Define and validate the collars

Each month settles on the 16th, rolled to the next USDZAR business day. Maturity is two business days before settlement. A USDZAR business day is open in both the United States and South Africa (`sjd.Calendar`). The put strike is 3% below the market forward; the call strike is 3% above it.

Strike rules stay in Python.

In [4]:
def make_option(maturity, settlement, strike, option_type, buy_sell):
    option = sjd.EuropeanOption(
        maturity=maturity,
        settlement=settlement,
        strike=strike,
        type=option_type,
        volume=volume,
        premium_payment_date=settlement,
        premium_amount=0.0,
        buy_sell=buy_sell,
    )
    option.validate(environment)
    if not option.is_ok():
        raise ValueError(option.error_messages())
    return option


collars = []
for settlement in settlements:
    maturity = holidays.advance(settlement, -2)
    atm = frs.get_forward(settlement)
    put_strike = round(atm * put_moneyness, 2)
    call_strike = round(atm * call_moneyness, 2)
    put = make_option(
        maturity, settlement, put_strike, sjd.PutCall.PUT, sjd.BuySell.BUY
    )
    call = make_option(
        maturity, settlement, call_strike, sjd.PutCall.CALL, sjd.BuySell.SELL
    )
    collars.append(
        {
            "maturity": maturity,
            "settlement": settlement,
            "atm": atm,
            "put_strike": put_strike,
            "call_strike": call_strike,
            "put": put,
            "call": call,
        }
    )

print(f"{'Settle':<12} {'Maturity':<12} {'ATM':>8} {'PutX':>8} {'CallX':>8}")
for collar in collars:
    print(
        f"{collar['settlement']:%Y-%m-%d} {collar['maturity']:%Y-%m-%d} "
        f"{collar['atm']:8.4f} {collar['put_strike']:8.2f} {collar['call_strike']:8.2f}"
    )

Settle       Maturity          ATM     PutX    CallX
2026-10-16 2026-10-14  17.5432    17.02    18.07
2026-11-16 2026-11-12  17.5880    17.06    18.12
2026-12-17 2026-12-14  17.6328    17.10    18.16
2027-01-19 2027-01-14  17.6807    17.15    18.21
2027-02-16 2027-02-11  17.7215    17.19    18.25
2027-03-16 2027-03-12  17.7623    17.23    18.30
2027-04-16 2027-04-14  17.8076    17.27    18.34
2027-05-17 2027-05-13  17.8530    17.32    18.39
2027-06-17 2027-06-14  17.8986    17.36    18.44
2027-07-16 2027-07-14  17.9413    17.40    18.48
2027-08-16 2027-08-12  17.9871    17.45    18.53
2027-09-16 2027-09-14  18.0330    17.49    18.57


In [5]:
print(
    f"{'Settle':<12} {'Put':>14} {'Call':>14} {'Collar':>14} "
    f"{'PutPrem':>10} {'CallPrem':>10}"
)
rows = []
for collar in collars:
    put_value = collar["put"].get_value(environment)
    call_value = collar["call"].get_value(environment)
    collar_value = put_value + call_value
    put_premium = collar["put"].get_option_value(environment) / volume
    call_premium = collar["call"].get_option_value(environment) / volume
    rows.append((collar, put_value, call_value, collar_value, put_premium, call_premium))
    print(
        f"{collar['settlement']:%Y-%m-%d} {put_value:14,.2f} {call_value:14,.2f} "
        f"{collar_value:14,.2f} {put_premium:10.4f} {call_premium:10.4f}"
    )

strip_value = sum(row[3] for row in rows)
print(f"\nTotal collar strip value: {strip_value:,.2f}")

Settle                  Put           Call         Collar    PutPrem   CallPrem
2026-10-16      56,198.81     -60,156.03      -3,957.22     0.0562     0.0602
2026-11-16     127,656.27    -134,694.83      -7,038.56     0.1277     0.1347
2026-12-17     194,413.40    -206,937.19     -12,523.78     0.1944     0.2069
2027-01-19     252,572.10    -265,954.26     -13,382.16     0.2526     0.2660
2027-02-16     299,023.48    -314,537.53     -15,514.05     0.2990     0.3145
2027-03-16     343,047.49    -357,304.52     -14,257.03     0.3430     0.3573
2027-04-16     387,513.99    -407,025.98     -19,511.99     0.3875     0.4070
2027-05-17     427,005.48    -444,438.25     -17,432.76     0.4270     0.4444
2027-06-17     464,039.26    -483,247.22     -19,207.96     0.4640     0.4832
2027-07-16     497,526.47    -519,793.63     -22,267.16     0.4975     0.5198
2027-08-16     530,717.39    -550,795.47     -20,078.08     0.5307     0.5508
2027-09-16     562,652.40    -588,146.77     -25,494.36     0.

In [6]:
# Check each leg against the bound Black-76 formula.
for collar, put_value, call_value, collar_value, _, _ in rows:
    settlement = collar["settlement"]
    maturity = collar["maturity"]
    forward = frs.get_forward(settlement)
    discount_factor = drs.get_discount_factor(settlement)
    standard_deviation = vrs.get_volatility(maturity) * math.sqrt(yf(maturity))

    put_formula = sjd.Black76Put(
        forward, collar["put_strike"], standard_deviation, discount_factor
    )
    call_formula = sjd.Black76Call(
        forward, collar["call_strike"], standard_deviation, discount_factor
    )
    if not put_formula.is_ok():
        raise ValueError(put_formula.error_message())
    if not call_formula.is_ok():
        raise ValueError(call_formula.error_message())

    expected_put = put_formula.premium() * volume
    expected_call = -call_formula.premium() * volume
    assert math.isclose(put_value, expected_put, rel_tol=1e-12)
    assert math.isclose(call_value, expected_call, rel_tol=1e-12)
    assert math.isclose(collar_value, expected_put + expected_call, rel_tol=1e-12)
    assert collar["put_strike"] < collar["atm"] < collar["call_strike"]

print("All 12 collars match Black-76 (purchased put + sold call).")

All 12 collars match Black-76 (purchased put + sold call).
